# Evaluación de Tavily como search tool de un agente LangGraph

## 1. Planteamiento

**Contexto.** La empresa quiere implementar un chatbot para consultas web. Se me encarga evaluar **Tavily** como `search_tool`.

**Objetivo.** Entender cómo se comporta Tavily dentro de un agente, cuánto cuesta en tokens y si se puede mejorar su coste y su relevancia.

**Plan de trabajo**

| Paso | Contenido |
|---|---|
| 1 | Comprobar que el agente busca cuando debe y solo entonces · acceder a los `ToolMessage` · medir `token_usage` y el peso de los `ToolMessage` |
| 2 | Entender el `search_tool`: estructura, snippets, score, parámetros, schema de `bind_tools()`, ¿el modelo usa los resultados o alucina? |
| 3 | Mejorar coste y relevancia: filtrar por score, no inyectar búsquedas infructuosas, prompt caching, relevancia evaluada por el LLM |

**Decisiones de diseño**

- Modelo: `claude-haiku-4-5` (corte de conocimiento: julio 2025).
- En vez de un *tool* para la fecha, el **system prompt** inyecta el corte de conocimiento y la fecha de hoy. Así el modelo sabe cuándo debe buscar sin gastar una llamada a una herramienta.
- La lógica está en `src/victorg_langgraph_websearchagent/agent_v2.py`; este notebook solo la usa y muestra resultados.

## 2. Preparación

In [1]:
%load_ext autoreload
%autoreload 2

import json

from victorg_langgraph_websearchagent.agent_v2 import (
    load_keys,
    build_search_agent,
    run_agent,
    get_answer,
    used_search,
    print_summary,
    get_tool_messages,
    print_tool_messages,
    count_tokens,
    print_tokens,
    toolmessage_tokens,
    print_toolmessage_weight,
    verify_toolmessage_tokens,
    print_weight_summary,
)

load_keys()
agent = build_search_agent()

## 3. Paso 1: comportamiento, acceso a ToolMessages y métricas de tokens

### 3.1 ¿El agente ejecuta la tool cuando debe (y solo entonces)?

Cada query lleva el comportamiento esperado: `expected_search=False` (conocimiento estático o fecha ya inyectada) o `True` (información actual).
Incluye casos límite cerca del corte (julio 2025) y una pregunta que comprueba la inyección de fecha (`¿Qué día es hoy?`, que **no** debe buscar).

In [2]:
queries = [
    # --- Estático / atemporal: NO debe buscar ---
    {"query": "¿Quién escribió Don Quijote de la Mancha?", "expected_search": False},
    {"query": "Explica qué es el teorema de Pitágoras", "expected_search": False},
    {"query": "¿Quién ganó el Mundial de fútbol de Qatar 2022?", "expected_search": False},
    {"query": "¿Qué es una lista por comprensión en Python?", "expected_search": False},
    {"query": "Hola, ¿qué puedes hacer?", "expected_search": False},
    # --- Fecha: está inyectada en el system prompt, NO debe buscar ---
    {"query": "¿Qué día es hoy?", "expected_search": False},
    # --- Herramienta multiply: NO debe buscar ---
    {"query": "Multiplica 123 por 456", "expected_search": False},
    # --- Información actual: DEBE buscar ---
    {"query": "¿Cuál es la última versión estable de LangGraph?", "expected_search": True},
    {"query": "¿Qué tiempo hace hoy en Madrid?", "expected_search": True},
    {"query": "¿Cuál es el precio actual del Bitcoin?", "expected_search": True},
    {"query": "Resume las noticias de hoy sobre inteligencia artificial", "expected_search": True},
    # --- Casos límite cerca del corte (julio 2025) ---
    {"query": "¿Quién ganó la Champions League 2024-2025?", "expected_search": False},
    {"query": "¿Quién ganó el Mundial de Clubes de la FIFA 2025?", "expected_search": True},
    {"query": "¿Quién ganó el Tour de Francia 2025?", "expected_search": True},
    # --- Mezcla: multiply + web ---
    {"query": "Multiplica 12 por 34 y dime cuál es la última versión estable de LangGraph", "expected_search": True},
]

In [3]:
results = [run_agent(agent, q["query"]) for q in queries]

In [4]:
# ¿Usó la web cuando debía? (✓ = coincide con lo esperado)
print_summary(queries, results)

#   Query                                                Esperado  Usó web   Tokens   Tool  OK
1   ¿Quién escribió Don Quijote de la Mancha?            estático  no          2717      0  ✓
2   Explica qué es el teorema de Pitágoras               estático  no          2960      0  ✓
3   ¿Quién ganó el Mundial de fútbol de Qatar 2022?      estático  no          2693      0  ✓
4   ¿Qué es una lista por comprensión en Python?         estático  no          3062      0  ✓
5   Hola, ¿qué puedes hacer?                             estático  no          2886      0  ✓
6   ¿Qué día es hoy?                                     estático  no          2583      0  ✓
7   Multiplica 123 por 456                               estático  no          5283     83  ✓
8   ¿Cuál es la última versión estable de LangGraph?     web       sí          7936   2457  ✓
9   ¿Qué tiempo hace hoy en Madrid?                      web       sí          8820   3481  ✓
10  ¿Cuál es el precio actual del Bitcoin?               we

### 3.2 Acceso a los ToolMessages

Un `ToolMessage` es el mensaje donde LangGraph guarda lo que devuelve una herramienta. Vive en `result["messages"]`.
`get_tool_messages(result)` los devuelve en una lista y `print_tool_messages(result)` los muestra legibles.

In [5]:
i = 7  # "¿Cuál es la última versión estable de LangGraph?"
print(queries[i]["query"], "\n")

# Acceso directo: lista de ToolMessage y contenido como diccionario
tool_messages = get_tool_messages(results[i])
tm = tool_messages[0]
tavily_output = json.loads(tm.content)

print("Herramienta:", tm.name)
print("tool_call_id:", tm.tool_call_id)
print("Claves de la salida de Tavily:", list(tavily_output.keys()))

¿Cuál es la última versión estable de LangGraph? 

Herramienta: tavily_search
tool_call_id: toolu_018jVQkZietV1NDtz5rH4vQg
Claves de la salida de Tavily: ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']


In [6]:
# Vista legible de todos los ToolMessages de esa ejecución
print_tool_messages(results[i])

ToolMessage 1: tavily_search
Argumentos: {'query': 'LangGraph última versión estable 2026', 'search_depth': 'basic'}
----------------------------------------------------------------------
Query: LangGraph última versión estable 2026
Tiempo de respuesta: 1.35 s
Nº de resultados: 5

[1] How to Build an AI Agent with LangGraph Python in 14 Steps [2026]
    URL:   https://tech-insider.org/langgraph-tutorial-ai-agent-python-2026
    Score: 0.8898454
    Texto: LangGraph Platform (formerly LangGraph Cloud) provides managed deployment for LangGraph applications. It handles scaling, persistence, and API serving so you can expose your agent as a production REST API without managing infrastructure. A production deployment guide from QWE.edu.pl cited LangGraph ...

[2] LangChain vs LangGraph (2026): When to Use Which
    URL:   https://www.ayautomate.com/blog/langgraph-vs-langchain
    Score: 0.8390101
    Texto: LangGraph hit v1.0 on October 22, 2025, alongside LangChain 1.0. Current version: 1.

### 3.3 Métricas de `token_usage` y peso de los ToolMessages

**Qué da la API.** Cada respuesta del modelo trae `usage_metadata` con `input_tokens`, `output_tokens` y `total_tokens` **de esa llamada**. La API **no** dice cuántos tokens tiene cada mensaje por separado. Hay que deducirlo.

**Cómo se determinan los tokens de un ToolMessage (no de un tool_call).**

En un agente con una herramienta, el modelo se llama dos veces. En cada llamada el modelo recibe **todo el contexto acumulado**:

```
Llamada 1   input I1 = system prompt + schemas de las tools + pregunta
            output O1 = el tool_call (el modelo pide la herramienta)

Llamada 2   input I2 = system prompt + schemas de las tools + pregunta
                       + el tool_call (el AIMessage de la llamada 1)
                       + el ToolMessage (resultado de la herramienta)
```

Restando una llamada a la otra, todo lo que se repite (system prompt, schemas, pregunta) se **cancela**:

```
I2 - I1 = tokens del tool_call (como input) + tokens del ToolMessage
```

El tool_call es lo que el modelo **generó** en la llamada 1, y su tamaño lo conocemos: es `O1`. Al volver a entrar como contexto ocupa esos mismos tokens. Por tanto:

```
tokens del ToolMessage = I2 - I1 - O1
```

**Diferencia entre tool_call y ToolMessage**

| | tool_call | ToolMessage |
|---|---|---|
| Quién lo escribe | el modelo | la herramienta (Tavily) |
| Cómo se factura | **output** la primera vez (`O1`) y luego input | solo **input** |
| Cómo se mide | directamente: `output_tokens` de la llamada 1 | por diferencia: `I2 - I1 - O1` |

**Dos matices importantes**

- **Se factura en cada llamada posterior.** El ToolMessage queda en el contexto. Si después de él el modelo se llama 2 veces más, esos tokens se pagan 2 veces como input. Por eso se calcula también el `input_facturado = tokens × nº de llamadas que lo arrastran`.
- **Es una deducción, no una medida directa.** Incluye unos pocos tokens de estructura (`tool_use` / `tool_result`) y puede haber una diferencia mínima al volver a codificar el tool_call. Para comprobarlo, `verify_toolmessage_tokens` cuenta con el endpoint `count_tokens` de Anthropic el texto puro del ToolMessage y lo compara.

**Peso del ToolMessage** = tokens de ToolMessage facturados / input total (o / tokens totales). El coste usa el precio de entrada de Haiku 4.5 (1 USD por millón de tokens).

In [7]:
# Datos crudos: usage_metadata de cada llamada al modelo
for m in results[i]["messages"]:
    usage = getattr(m, "usage_metadata", None)
    if usage:
        print(usage)

{'input_tokens': 2564, 'output_tokens': 84, 'total_tokens': 2648, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}
{'input_tokens': 5021, 'output_tokens': 267, 'total_tokens': 5288, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}


In [8]:
# Desglose por llamada + tokens del tool_call
print_tokens(results[i])

Llamada 1 (tavily_search): input=2564 | output=84 | total=2648
Llamada 2 (respuesta final): input=5021 | output=267 | total=5288
TOTAL: input=7585 | output=351 | total=7936
TOOL CALL: petición=84 | resultado=2373 | total herramienta=2457


In [9]:
# Tokens del ToolMessage con la cuenta paso a paso (I2 - I1 - O1)
print_toolmessage_weight(results[i])

Ronda 1 (tavily_search)
  I1 (input antes del tool_call)      = 2564
  O1 (output: el tool_call)           = 84
  I2 (input con el ToolMessage)       = 5021
  ToolMessage = I2 - I1 - O1          = 2373 tokens (47% del input de la llamada siguiente)
  Llamadas que lo arrastran: 1 -> input facturado por el ToolMessage = 2373
TOTAL ToolMessages: 2373 tokens | 31% del input total | 30% de los tokens totales | coste ~ $0.00237


In [10]:
# Comprobación independiente con el endpoint count_tokens de Anthropic
verify_toolmessage_tokens(results[i])

Ronda 1: por diferencia (I2-I1-O1) = 2373 | texto del ToolMessage con count_tokens = 2361 | diferencia = 12
(la diferencia son los tokens de la estructura tool_use/tool_result, pocos y constantes)


In [11]:
# Peso de los ToolMessages en todas las queries que usaron herramientas
print_weight_summary(queries, results)

#   Query                                           TM tok  Facturado  % input       USD
7   Multiplica 123 por 456                              14         14       0%   0.00001
8   ¿Cuál es la última versión estable de Lan...      2373       2373      31%   0.00237
9   ¿Qué tiempo hace hoy en Madrid?                   3404       3404      40%   0.00340
10  ¿Cuál es el precio actual del Bitcoin?            2775       2775      35%   0.00278
11  Resume las noticias de hoy sobre intelige...      1559       1559      23%   0.00156
12  ¿Quién ganó la Champions League 2024-2025?        2466       2466      32%   0.00247
13  ¿Quién ganó el Mundial de Clubes de la FI...      2744       2744      34%   0.00274
14  ¿Quién ganó el Tour de Francia 2025?              2350       2350      31%   0.00235
15  Multiplica 12 por 34 y dime cuál es la úl...      2171       2171      29%   0.00217


In [12]:
# Medias (solo queries con búsqueda web)
weights = [toolmessage_tokens(r) for r in results]
searched_idx = [k for k, w in enumerate(weights) if w["rondas"]]

if searched_idx:
    avg_tokens = sum(weights[k]["toolmessage_tokens"] for k in searched_idx) / len(searched_idx)
    avg_weight = sum(weights[k]["peso_en_input_total"] for k in searched_idx) / len(searched_idx)
    avg_cost = sum(weights[k]["coste_usd"] for k in searched_idx) / len(searched_idx)
    print(f"Queries con herramientas: {len(searched_idx)} de {len(queries)}")
    print(f"Tokens medios de ToolMessage: {avg_tokens:.0f}")
    print(f"Peso medio en el input total: {avg_weight:.0%}")
    print(f"Coste medio de los ToolMessages: ${avg_cost:.5f}")

Queries con herramientas: 9 de 15
Tokens medios de ToolMessage: 2206
Peso medio en el input total: 28%
Coste medio de los ToolMessages: $0.00221


## 4. Resultados y conclusiones del Paso 1

*(Completar con los números reales tras ejecutar el notebook.)*

- **Comportamiento:** ¿en cuántas queries coincidió lo esperado con lo que hizo el agente? ¿Qué casos fallaron (✗) y qué tienen en común?
- **Fecha:** ¿respondió `¿Qué día es hoy?` sin buscar? ¿Funciona la inyección de fecha en el system prompt?
- **Peso de los ToolMessages:** tokens medios de un ToolMessage de Tavily, % del input total y coste medio por búsqueda.
- **Comprobación:** ¿cuánto difiere `I2 - I1 - O1` del recuento directo con `count_tokens`?

## 5. Siguientes pasos

- **Paso 2:** entender la estructura de lo que devuelve Tavily, los snippets, el score y el schema que el modelo ve con `bind_tools()`.
- **Paso 3:** probar mejoras de coste y relevancia (filtrar por score, no inyectar búsquedas infructuosas, prompt caching, evaluación de relevancia con un schema pydantic).